# schema

> Proposed Kimball-style PostgreSQL warehouse schema for Bitcoin block, transaction, fee, and network analytics.

## Design intent

This schema is organized as a dimensional warehouse, following the Kimball approach: declare the business processes, declare the grain, identify conformed dimensions, then design fact tables around measurements at that grain.

The goal is to support dashboard views like block lists, block detail panels, fee distributions, reward breakdowns, capacity charts, transaction mix, output type mix, value flow, and network status trends.

The warehouse should preserve raw Bitcoin Core observations where useful, but present analytics through stable dimensions and facts rather than direct RPC-shaped tables.

## Dimensional bus matrix

| Business process | Primary grain | Core fact table | Conformed dimensions |
| --- | --- | --- | --- |
| Block production | One canonical-chain block | `fact_block` | `dim_date`, `dim_time`, `dim_block`, `dim_mining_pool`, `dim_chain_state` |
| Transaction confirmation | One transaction in one block | `fact_transaction` | `dim_date`, `dim_time`, `dim_block`, `dim_transaction`, `dim_script_type`, `dim_fee_band` |
| Transaction inputs | One input spent by one transaction | `fact_tx_input` | `dim_date`, `dim_time`, `dim_block`, `dim_transaction`, `dim_prevout`, `dim_script_type` |
| Transaction outputs | One output created by one transaction | `fact_tx_output` | `dim_date`, `dim_time`, `dim_block`, `dim_transaction`, `dim_address`, `dim_script_type` |
| Fee distribution | One fee-rate bucket per block | `fact_block_fee_bucket` | `dim_date`, `dim_time`, `dim_block`, `dim_fee_band` |
| Network status snapshots | One observation interval | `fact_network_snapshot` | `dim_date`, `dim_time`, `dim_observation_source` |
| Mempool snapshots | One mempool bucket per observation interval | `fact_mempool_fee_bucket` | `dim_date`, `dim_time`, `dim_fee_band`, `dim_observation_source` |
| Market snapshots | One market quote per observation interval | `fact_market_snapshot` | `dim_date`, `dim_time`, `dim_asset`, `dim_currency`, `dim_observation_source` |
| Chain reorganizations | One block version observed per height | `fact_block_observation` | `dim_date`, `dim_time`, `dim_block`, `dim_chain_state`, `dim_observation_source` |

## Grain rules

Every fact table must have a declared grain before columns are added. Avoid mixing block-level, transaction-level, and output-level measures in the same fact table unless they share the exact same grain.

| Fact table | Grain declaration |
| --- | --- |
| `fact_block` | Exactly one row per block hash that is currently considered canonical, plus a durable block dimension reference. |
| `fact_transaction` | Exactly one row per transaction confirmed in a specific block. A transaction that appears in a later reorged-out block should be represented through observation/versioning, not overwritten silently. |
| `fact_tx_input` | Exactly one row per transaction input. Coinbase inputs are included with an `is_coinbase` flag and no previous outpoint reference. |
| `fact_tx_output` | Exactly one row per transaction output. Unspendable outputs are retained and classified by script type. |
| `fact_block_fee_bucket` | Exactly one row per block per fee-rate bucket. Buckets should be deterministic for reproducible visualizations. |
| `fact_network_snapshot` | Exactly one row per observation timestamp and source. |
| `fact_mempool_fee_bucket` | Exactly one row per observation timestamp, source, and fee-rate bucket. |
| `fact_market_snapshot` | Exactly one row per observation timestamp, asset, quote currency, and source. |

## Conformed dimensions

### `dim_date`

Calendar dimension shared by all facts.

| Column | Type intent | Notes |
| --- | --- | --- |
| `date_key` | integer surrogate key | Format `YYYYMMDD`. |
| `date_actual` | date | UTC date unless a separate local reporting calendar is later needed. |
| `year`, `quarter`, `month`, `day_of_month`, `day_of_week` | integer/text attributes | Useful for rollups. |
| `is_weekend` | boolean | Convenience attribute. |

### `dim_time`

Time-of-day dimension shared by facts that need intraday grouping.

| Column | Type intent | Notes |
| --- | --- | --- |
| `time_key` | integer surrogate key | Format `HHMMSS`. |
| `time_actual` | time | UTC time. |
| `hour`, `minute`, `second` | integer attributes | Enables hourly/minute rollups. |
| `minute_of_day` | integer | Useful for intraday charts. |

### `dim_block`

Durable block descriptor. Keep block identity here and measurements in facts.

| Column | Type intent | Notes |
| --- | --- | --- |
| `block_key` | bigint surrogate key | Warehouse surrogate key. |
| `block_hash` | text/natural key | 64-character display-order hash from Bitcoin Core RPC. |
| `height` | integer | Natural chain height. Not globally unique across competing block versions. |
| `version` | integer | Header version. |
| `previous_block_hash` | text | Parent block hash. |
| `merkle_root` | text | Transaction merkle root. |
| `witness_commitment` | text nullable | Present when available. |
| `bits` | text | Compact difficulty representation. |
| `nonce` | bigint | Header nonce. |
| `block_time_utc` | timestamptz | Timestamp from block header. |

### `dim_transaction`

Transaction identity and stable decoded attributes.

| Column | Type intent | Notes |
| --- | --- | --- |
| `transaction_key` | bigint surrogate key | Warehouse surrogate key. |
| `txid` | text/natural key | Legacy transaction id. |
| `wtxid` | text nullable | Witness transaction id where available. |
| `version` | integer | Transaction version. |
| `lock_time` | bigint | Raw locktime. |
| `is_coinbase` | boolean | True for block subsidy transaction. |
| `has_witness` | boolean | True when witness data is present. |

### `dim_script_type`

Conformed output/input script classification.

| Column | Type intent | Notes |
| --- | --- | --- |
| `script_type_key` | smallint surrogate key | Small static dimension. |
| `script_type_code` | text | Examples: `p2pkh`, `p2sh`, `p2wpkh`, `p2wsh`, `p2tr`, `op_return`, `multisig`, `nonstandard`, `unknown`. |
| `script_family` | text | Legacy, nested SegWit, native SegWit, Taproot, data, unknown. |
| `is_spendable` | boolean | False for provably unspendable outputs such as `OP_RETURN`. |

### `dim_address`

Address-like recipient identity. This should be optional because not every script maps cleanly to an address.

| Column | Type intent | Notes |
| --- | --- | --- |
| `address_key` | bigint surrogate key | Warehouse surrogate key. |
| `address` | text nullable | Decoded address when available. |
| `network` | text | `mainnet`, `testnet`, `signet`, `regtest`. |
| `script_type_key` | smallint foreign key | Links to `dim_script_type`. |
| `address_hash` | text nullable | Optional normalized hash/program for grouping. |

### `dim_fee_band`

Stable fee-rate buckets for charts and filtering.

| Column | Type intent | Notes |
| --- | --- | --- |
| `fee_band_key` | integer surrogate key | Static or slowly revised dimension. |
| `lower_sat_vb` | numeric | Inclusive lower bound. |
| `upper_sat_vb` | numeric nullable | Exclusive upper bound, null for open-ended. |
| `band_label` | text | Examples: `1-2 sat/vB`, `25-30 sat/vB`, `200+ sat/vB`. |
| `sort_order` | integer | Deterministic chart ordering. |

### `dim_mining_pool`

Slowly changing dimension for coinbase tag attribution.

| Column | Type intent | Notes |
| --- | --- | --- |
| `mining_pool_key` | bigint surrogate key | Warehouse surrogate key. |
| `pool_name` | text | Display name such as `AntPool`, `Foundry USA`, `Unknown`. |
| `coinbase_tag_pattern` | text nullable | Attribution pattern used during classification. |
| `valid_from_utc`, `valid_to_utc` | timestamptz | Type 2 range for attribution changes. |
| `is_current` | boolean | Current attribution row. |

### `dim_chain_state`

Small dimension describing how a block observation relates to the active chain.

| Column | Type intent | Notes |
| --- | --- | --- |
| `chain_state_key` | smallint surrogate key | Static dimension. |
| `chain_state_code` | text | `canonical`, `stale`, `orphan_observed`, `candidate`, `unknown`. |
| `is_active_chain` | boolean | True for canonical active-chain blocks. |

### `dim_observation_source`

Tracks source and cadence of observed metrics.

| Column | Type intent | Notes |
| --- | --- | --- |
| `observation_source_key` | smallint surrogate key | Static or slowly changing dimension. |
| `source_name` | text | Examples: local bitcoind, external fee API, exchange API. |
| `source_type` | text | RPC, API, derived, manual. |
| `network` | text | Mainnet/test network context. |

### `dim_asset` and `dim_currency`

Small dimensions for market conversion and fiat-denominated dashboard values.

| Dimension | Key columns | Notes |
| --- | --- | --- |
| `dim_asset` | `asset_key`, `asset_symbol`, `asset_name` | Usually `BTC`, but this keeps the market facts extensible. |
| `dim_currency` | `currency_key`, `currency_code`, `currency_name` | Usually `USD` for first dashboard version. |

## Core fact tables

### `fact_block`

One row per canonical-chain block. This powers block list cards, block detail summaries, capacity, rewards, and high-level fee metrics.

| Column | Type intent | Additivity | Notes |
| --- | --- | --- | --- |
| `block_key` | bigint FK | n/a | Links to `dim_block`. |
| `date_key`, `time_key` | integer FK | n/a | Based on block header time. |
| `mining_pool_key` | bigint FK | n/a | Derived from coinbase attribution. |
| `chain_state_key` | smallint FK | n/a | Usually canonical in this fact. |
| `height` | integer degenerate dimension | n/a | Repeated for convenient partitioning/filtering. |
| `tx_count` | integer | additive | Number of transactions. |
| `input_count`, `output_count` | integer | additive | Decoded transaction counts. |
| `block_size_bytes` | integer | additive | Serialized size. |
| `block_weight_wu` | integer | additive | Block weight units. |
| `block_vsize_vb` | integer | additive | Virtual size. |
| `capacity_used_ratio` | numeric | semi-additive | `block_weight_wu / 4,000,000`. |
| `subsidy_btc` | numeric | additive | Coinbase subsidy. |
| `total_fees_btc` | numeric | additive | Sum of transaction fees excluding subsidy. |
| `total_reward_btc` | numeric | additive | Subsidy plus fees. |
| `median_fee_sat_vb`, `p10_fee_sat_vb`, `p25_fee_sat_vb`, `p50_fee_sat_vb`, `p75_fee_sat_vb`, `p90_fee_sat_vb`, `p99_fee_sat_vb` | numeric | non-additive | Distribution metrics for dashboard labels. |
| `min_fee_sat_vb`, `max_fee_sat_vb` | numeric | non-additive | Range metrics. |
| `clearing_fee_sat_vb` | numeric | non-additive | Lowest non-coinbase confirmed fee rate or selected policy metric. |
| `segwit_tx_count`, `taproot_tx_count` | integer | additive | Confirmation mix. |
| `op_return_output_count` | integer | additive | Data output count. |
| `estimated_value_settled_btc` | numeric | additive with caution | Best-effort economic volume after change heuristic. |
| `coin_days_destroyed` | numeric | additive | Requires previous output age/value. |
| `oldest_input_age_days` | numeric | non-additive | Max age among spent inputs. |
| `average_input_age_days` | numeric | non-additive | Weighted or unweighted, define explicitly. |

### `fact_transaction`

One row per confirmed transaction in one block. This supports largest transactions, highest fee, highest fee rate, transaction mix, and per-block distribution charts.

| Column | Type intent | Additivity | Notes |
| --- | --- | --- | --- |
| `transaction_key` | bigint FK | n/a | Links to `dim_transaction`. |
| `block_key` | bigint FK | n/a | Confirmation block. |
| `date_key`, `time_key` | integer FK | n/a | Block time. |
| `position_in_block` | integer | n/a | Coinbase is usually `0`. |
| `input_count`, `output_count` | integer | additive | Per transaction counts. |
| `size_bytes`, `stripped_size_bytes`, `weight_wu`, `vsize_vb` | integer | additive | Transaction size measures. |
| `input_value_btc` | numeric nullable | additive | Null or zero for coinbase, depending on semantic convention. |
| `output_value_btc` | numeric | additive | Sum of outputs. |
| `fee_btc` | numeric nullable | additive | Null for coinbase. |
| `fee_sat_vb` | numeric nullable | non-additive | `fee_sats / vsize_vb`. |
| `is_coinbase`, `is_rbf_signaling`, `has_witness`, `has_op_return` | boolean | n/a | Transaction flags. |
| `dominant_input_script_type_key` | smallint FK nullable | n/a | Optional classification shortcut. |
| `dominant_output_script_type_key` | smallint FK nullable | n/a | Optional classification shortcut. |
| `estimated_change_value_btc` | numeric nullable | additive with caution | Heuristic-derived. |
| `estimated_payment_value_btc` | numeric nullable | additive with caution | Useful for economic signal dashboards. |

### `fact_tx_input`

One row per transaction input. This is the bridge to previous outputs and enables coin days destroyed, age metrics, consolidation detection, and value-flow analysis.

| Column | Type intent | Additivity | Notes |
| --- | --- | --- | --- |
| `transaction_key` | bigint FK | n/a | Spending transaction. |
| `block_key` | bigint FK | n/a | Spending block. |
| `date_key`, `time_key` | integer FK | n/a | Spending block time. |
| `input_index` | integer degenerate dimension | n/a | Input position. |
| `prev_txid` | text nullable | n/a | Degenerate dimension for previous outpoint. |
| `prev_output_index` | integer nullable | n/a | Previous vout. |
| `prevout_value_btc` | numeric nullable | additive | Requires txindex, block undo data, or verbosity 3 prevouts. |
| `prevout_block_key` | bigint FK nullable | n/a | Creation block if known. |
| `prevout_age_blocks` | integer nullable | non-additive | Spend height minus create height. |
| `prevout_age_seconds` | bigint nullable | non-additive | Spend time minus create time. |
| `coin_days_destroyed` | numeric nullable | additive | `prevout_value_btc * age_days`. |
| `sequence` | bigint | n/a | Raw sequence. |
| `is_coinbase` | boolean | n/a | Coinbase input flag. |
| `is_rbf_signaling` | boolean | n/a | Sequence below final threshold. |

### `fact_tx_output`

One row per transaction output. This powers output type mix, value distribution, OP_RETURN analysis, and future UTXO-style views.

| Column | Type intent | Additivity | Notes |
| --- | --- | --- | --- |
| `transaction_key` | bigint FK | n/a | Creating transaction. |
| `block_key` | bigint FK | n/a | Creating block. |
| `date_key`, `time_key` | integer FK | n/a | Creating block time. |
| `output_index` | integer degenerate dimension | n/a | Vout position. |
| `address_key` | bigint FK nullable | n/a | Null when no address maps cleanly. |
| `script_type_key` | smallint FK | n/a | Output script classification. |
| `value_btc` | numeric | additive | Output value. |
| `script_size_bytes` | integer | additive | ScriptPubKey byte length. |
| `is_spendable` | boolean | n/a | Derived from script classification. |
| `is_dust` | boolean | n/a | Based on policy threshold at observation time. |
| `is_likely_change` | boolean nullable | n/a | Heuristic-derived. |
| `spent_transaction_key` | bigint FK nullable | n/a | Optional denormalized convenience once spent. |
| `spent_block_key` | bigint FK nullable | n/a | Optional denormalized convenience once spent. |

## Distribution and snapshot facts

### `fact_block_fee_bucket`

One row per block per fee-rate bucket. This supports the histogram in the block detail view and miniature bar summaries in block cards.

| Column | Type intent | Additivity | Notes |
| --- | --- | --- | --- |
| `block_key` | bigint FK | n/a | Confirmation block. |
| `date_key`, `time_key` | integer FK | n/a | Block time. |
| `fee_band_key` | integer FK | n/a | Bucket definition. |
| `tx_count` | integer | additive | Non-coinbase transactions in bucket. |
| `total_vsize_vb` | integer | additive | Sum of vsize in bucket. |
| `total_fees_btc` | numeric | additive | Sum of fees in bucket. |
| `mean_fee_sat_vb` | numeric | non-additive | Weighted by vsize if used for visualization color. |
| `min_fee_sat_vb`, `max_fee_sat_vb` | numeric | non-additive | Bucket range observed. |

### `fact_mempool_fee_bucket`

One row per observation time, source, and fee-rate bucket. This supports next-block threshold and mempool fee histogram views.

| Column | Type intent | Additivity | Notes |
| --- | --- | --- | --- |
| `observed_at_utc` | timestamptz degenerate dimension | n/a | Exact snapshot timestamp. |
| `date_key`, `time_key` | integer FK | n/a | Snapshot time. |
| `observation_source_key` | smallint FK | n/a | Usually local bitcoind. |
| `fee_band_key` | integer FK | n/a | Fee-rate bucket. |
| `tx_count` | integer | additive | Mempool txs in bucket. |
| `total_vsize_vb` | bigint | additive | Mempool virtual bytes in bucket. |
| `total_fees_btc` | numeric | additive | Total fees in bucket. |
| `min_fee_sat_vb`, `max_fee_sat_vb`, `mean_fee_sat_vb` | numeric | non-additive | Observed bucket stats. |

### `fact_network_snapshot`

One row per observation timestamp and source. This powers the network status strip: hash rate, difficulty, block interval, supply, and fee trends.

| Column | Type intent | Additivity | Notes |
| --- | --- | --- | --- |
| `observed_at_utc` | timestamptz degenerate dimension | n/a | Exact observation timestamp. |
| `date_key`, `time_key` | integer FK | n/a | Observation time. |
| `observation_source_key` | smallint FK | n/a | Source identity. |
| `best_block_key` | bigint FK nullable | n/a | Tip at observation time. |
| `best_height` | integer | n/a | Tip height. |
| `difficulty` | numeric | non-additive | Current difficulty. |
| `estimated_hashrate_eh_s` | numeric | non-additive | Windowed estimate. |
| `median_block_interval_seconds` | numeric | non-additive | Windowed block time metric. |
| `mempool_tx_count` | integer | semi-additive | Point-in-time count. |
| `mempool_vsize_vb` | bigint | semi-additive | Point-in-time size. |
| `recommended_fee_fast_sat_vb`, `recommended_fee_half_hour_sat_vb`, `recommended_fee_hour_sat_vb`, `recommended_fee_minimum_sat_vb` | numeric | non-additive | Source-defined estimates. |
| `circulating_supply_btc` | numeric | non-additive | Supply at tip. |

### `fact_market_snapshot`

One row per observed asset/currency quote. This lets BTC-denominated measures be displayed in fiat without baking exchange rates into blockchain facts.

| Column | Type intent | Additivity | Notes |
| --- | --- | --- | --- |
| `observed_at_utc` | timestamptz degenerate dimension | n/a | Quote timestamp. |
| `date_key`, `time_key` | integer FK | n/a | Quote time. |
| `asset_key`, `currency_key` | smallint FK | n/a | Example: BTC/USD. |
| `observation_source_key` | smallint FK | n/a | Exchange or aggregator. |
| `price` | numeric | non-additive | Last or reference price. |
| `volume_24h` | numeric nullable | non-additive | Optional market context. |

## Reorg and history handling

Bitcoin chain state can change near the tip, so the warehouse should distinguish immutable observations from the current canonical presentation.

| Table | Purpose |
| --- | --- |
| `fact_block_observation` | Append-only record of every block hash observed at a height, including when it was first seen, last seen, and whether it was active at observation time. |
| `fact_block` | Current canonical analytic fact for ordinary dashboards. This can be rebuilt from observations after a reorg. |
| `dim_chain_state` | Conformed classification for canonical, stale, candidate, and unknown states. |

`fact_block_observation` recommended fields: `block_key`, `height`, `chain_state_key`, `observed_at_utc`, `first_seen_at_utc`, `last_seen_at_utc`, `confirmations_at_observation`, `best_chain_tip_hash`, `observation_source_key`.

For dashboards, default to canonical facts but retain stale observations for auditability, reorg analysis, and correctness around near-tip data.

## Derived aggregate tables

These are optional summary stars or materialized views. They should be rebuildable from atomic facts.

| Aggregate | Grain | Use case |
| --- | --- | --- |
| `agg_block_recent_metrics` | One row per recent canonical block | Fast block list cards and sparkline inputs. |
| `agg_daily_chain_metrics` | One row per UTC date | Daily fees, rewards, volume, transaction count, supply, average block interval. |
| `agg_hourly_fee_metrics` | One row per UTC hour | Fee trend charts and network status strips. |
| `agg_block_output_type_mix` | One row per block and script type | Donut charts for output type share. |
| `agg_block_transaction_mix` | One row per block and transaction class | Consolidation, batched payout, RBF replacement, dust, whale transaction counts. |
| `agg_address_activity_daily` | One row per address and UTC date | Future address-level analytics. Use cautiously because address clustering is not identity. |

## Dashboard mapping

| Dashboard element | Primary source |
| --- | --- |
| Recent block list | `fact_block`, `agg_block_recent_metrics`, `dim_block`, `dim_mining_pool` |
| Block fee distribution histogram | `fact_block_fee_bucket`, `dim_fee_band` |
| Capacity panel | `fact_block.block_weight_wu`, `fact_block.block_vsize_vb`, `fact_block.capacity_used_ratio` |
| Rewards panel | `fact_block.subsidy_btc`, `fact_block.total_fees_btc`, `fact_block.total_reward_btc` |
| Largest transaction and highest fee rows | `fact_transaction` filtered by `block_key` |
| Economic signals | `fact_block`, `fact_tx_input`, `fact_transaction` |
| Transaction mix | `agg_block_transaction_mix` or derived from `fact_transaction` |
| Output type donut | `agg_block_output_type_mix` or derived from `fact_tx_output` and `dim_script_type` |
| Mempool histogram | `fact_mempool_fee_bucket`, `dim_fee_band` |
| Recommended fee | `fact_network_snapshot` or a dedicated fee estimate snapshot if source semantics diverge |
| Hash rate, difficulty, block time | `fact_network_snapshot`, `fact_block` rolling windows |
| Fiat-denominated values | Blockchain facts joined to nearest appropriate `fact_market_snapshot` |

## PostgreSQL physical design notes

PostgreSQL can serve both the atomic warehouse and early dashboard workloads if the large facts are partitioned and indexed around common access paths.

| Area | Recommendation |
| --- | --- |
| Surrogate keys | Use generated `bigint` identity keys for large dimensions. Keep Bitcoin hashes as unique natural keys. |
| Numeric BTC values | Store BTC as `numeric(20, 8)` for readability, or store sats as `bigint` for exactness. Prefer sats internally for high-volume facts, with BTC views for presentation. |
| Hash columns | Use fixed-length text or `bytea`. Text is easier during exploration; `bytea` can be more compact. Pick one convention and keep display-order/hash-byte-order semantics documented. |
| Partitioning | Partition `fact_transaction`, `fact_tx_input`, and `fact_tx_output` by block height range or block date. Height-range partitioning aligns well with chain ingestion. |
| Indexes | Index natural lookup paths: `dim_block(block_hash)`, `dim_block(height)`, `dim_transaction(txid)`, `fact_transaction(block_key)`, `fact_tx_output(transaction_key)`, `fact_tx_input(transaction_key)`. |
| Recent dashboard queries | Add covering indexes or materialized aggregates for recent blocks, because most UI traffic will repeatedly scan the tip window. |
| Append-only staging | Keep raw RPC payloads in staging tables or object storage for replay/debugging, but do not make them the primary analytics interface. |
| Rebuildability | Treat aggregates as disposable. Atomic dimensions and facts are the durable source for analytic rebuilds. |
| Time zones | Store observation and block timestamps as UTC. Apply local time zones only in presentation. |

## Staging layer

A thin staging layer keeps ingestion honest and lets the dimensional model evolve without losing raw observations.

| Staging table | Grain | Notes |
| --- | --- | --- |
| `stg_rpc_block_raw` | One RPC block response per block hash and verbosity/source | Store block hash, height if known, verbosity, payload JSON or raw hex, observed timestamp, source. |
| `stg_rpc_transaction_raw` | One RPC transaction response per txid/source | Useful when transactions are fetched independently. |
| `stg_mempool_snapshot_raw` | One raw mempool observation per timestamp/source | Can be large; retain according to storage budget. |
| `stg_fee_estimate_raw` | One fee estimate response per timestamp/source | Captures source-specific fee semantics. |
| `stg_market_quote_raw` | One market response per timestamp/source | Keeps exchange-specific response fields outside conformed facts. |

## Semantic conventions

| Topic | Convention |
| --- | --- |
| Canonical chain | User-facing dashboards read canonical facts by default. Stale/reorged blocks remain queryable through observation facts. |
| Coinbase fees | Coinbase transaction `fee_btc` should be null, while block-level `total_fees_btc` is measured from non-coinbase transactions and reward decomposition. |
| Fee rate | Use sat/vB as the standard unit. Store exact fee sats and vsize where possible, derive fee rate for presentation and bucketing. |
| Value settled | Keep raw output value separate from heuristic economic value. Label heuristic measures explicitly. |
| Address identity | Address is not user identity. Any clustering/entity model should be a later optional dimension with clear caveats. |
| Dust and standardness | Policy-derived classifications can change over time. Include policy version/source where these classifications become important. |
| Market conversion | Fiat values should join to quotes at query or aggregate time, not overwrite native BTC/sat facts. |

## Initial build order

A practical first increment should focus on block and transaction analytics before deeper mempool and address behavior.

1. Build staging for `getblock` verbosity 2 or 3 responses.
2. Populate `dim_date`, `dim_time`, `dim_block`, `dim_transaction`, and `dim_script_type`.
3. Populate `fact_block`, `fact_transaction`, and `fact_block_fee_bucket`.
4. Add `fact_tx_input` and `fact_tx_output` once previous output values and script classifications are reliable.
5. Add mempool and network snapshot facts for the top dashboard strip.
6. Add aggregate tables/materialized views only after the atomic grain is stable and query patterns are visible.